# Asteroid Cost Atlas — Mission Economics Explorer

Economic analysis of 1.5M asteroids for selective precious metal extraction.
Prices: April 2026 spot. Cost: $300M mission + Falcon Heavy transport.

**Data sources:** SBDB (1.5M objects), LCDB (31K rotation periods), NEOWISE (164K diameters/albedos), SDSS MOC (40K color indices), JPL Horizons (NEA high-precision elements).

**Prerequisite:** Run `make pipeline` before using this notebook.

In [ ]:
from pathlib import Path
from asteroid_cost_atlas.utils.query import CostAtlasDB

processed = Path("../data/processed")
for pattern in ("atlas_*.parquet", "sbdb_physical_*.parquet"):
    candidates = sorted(processed.glob(pattern))
    if candidates:
        break

if not candidates:
    raise FileNotFoundError("No processed parquet found. Run 'make pipeline' first.")

db = CostAtlasDB(candidates[-1])
cols = db.sql('SELECT * FROM atlas LIMIT 0').columns.tolist()
print(f"Loaded: {candidates[-1].name} ({len(cols)} columns)")

In [ ]:
STD_COLS = """
    economic_priority_rank AS rank,
    name,
    composition_class AS class,
    ROUND(diameter_estimated_km, 3) AS diameter_km,
    ROUND(delta_v_km_s, 2) AS delta_v,
    ROUND(margin_per_kg, 0) AS margin_per_kg,
    ROUND(break_even_kg, 0) AS break_even_kg,
    ROUND(total_extractable_precious_kg, 1) AS extractable_kg,
    is_viable,
    ROUND(missions_supported, 0) AS missions,
    ROUND(mission_profit_usd, 0) AS per_mission_profit,
    ROUND(campaign_profit_usd, 0) AS campaign_profit,
    ROUND(total_precious_value_usd, 0) AS asteroid_value_usd,
    ROUND(moid_au, 4) AS moid_au,
    neo,
    orbit_class
"""
print("Standard columns loaded")

---
## Reference

### Metal prices (April 2, 2026)

| Metal | $/kg | Source |
|---|---|---|
| Rhodium | $299,000 | Kitco |
| Iridium | $254,000 | DailyMetalPrice |
| Gold | $150,740 | Kitco |
| Platinum | $63,300 | Kitco |
| Ruthenium | $56,260 | DailyMetalPrice |
| Palladium | $47,870 | Kitco |
| Osmium | $12,860 | est. |

Weighted specimen value: ~$90K/kg refined concentrate.

### Mission cost

```
total_fixed = $300M + 1t system × transport_per_kg
mission_cost = total_fixed + payload × (transport + $5K extraction)
break_even = total_fixed / margin_per_kg
```

Each mission carries ≥ break_even_kg payload → **every mission is profitable by construction.**
Leftover material below break_even is not worth a separate mission.

### Column guide

| Column | Description |
|---|---|
| `margin_per_kg` | Specimen value − transport − extraction overhead |
| `break_even_kg` | Total fixed cost / margin. Min payload for profitable mission |
| `is_viable` | Asteroid has ≥ break_even_kg extractable precious metals |
| `missions` | Number of profitable missions (each ≥ break_even payload) |
| `per_mission_profit` | Revenue − cost for one mission |
| `campaign_profit` | Sum of all mission profits |

---
## 1. Overview

In [ ]:
db.sql("""
    SELECT
        COUNT(*) AS total,
        COUNTIF(margin_per_kg > 0) AS positive_margin,
        COUNTIF(is_viable) AS viable,
        ROUND(SUM(CASE WHEN is_viable THEN missions_supported ELSE 0 END), 0) AS total_missions,
        ROUND(SUM(CASE WHEN is_viable THEN campaign_profit_usd ELSE 0 END), 0) AS total_campaign_profit,
        ROUND(MAX(campaign_profit_usd), 0) AS best_campaign_profit
    FROM atlas
""")

---
## 1b. Data Source Coverage

How much did each external source contribute to the enrichment pipeline?

In [ ]:
db.sql("""
    SELECT
        -- Diameter provenance
        COUNTIF(diameter_source = 'measured') AS diameter_sbdb_measured,
        COUNTIF(diameter_source = 'neowise') AS diameter_neowise,
        COUNTIF(diameter_source = 'estimated') AS diameter_h_estimated,
        COUNTIF(diameter_source IS NULL) AS diameter_missing,
        -- Rotation provenance
        COUNTIF(rotation_source = 'sbdb') AS rotation_sbdb,
        COUNTIF(rotation_source = 'lcdb') AS rotation_lcdb,
        COUNTIF(rotation_source IS NULL) AS rotation_missing,
        -- Orbital precision
        COUNTIF(orbital_precision_source = 'horizons') AS orbital_horizons,
        COUNTIF(orbital_precision_source = 'sbdb') AS orbital_sbdb,
        -- Composition provenance
        COUNTIF(composition_source = 'taxonomy') AS comp_taxonomy,
        COUNTIF(composition_source = 'sdss_colors') AS comp_sdss,
        COUNTIF(composition_source = 'albedo') AS comp_albedo,
        COUNTIF(composition_source = 'none') AS comp_unknown
    FROM atlas
""")

---
## 2. Most Profitable Campaigns

In [ ]:
db.sql(f"""
    SELECT {STD_COLS}
    FROM atlas
    WHERE is_viable
    ORDER BY campaign_profit_usd DESC
    LIMIT 30
""")

---
## 3. Per-Mission Economics — Top Targets

Revenue, cost, and profit for a single mission to each target.

In [ ]:
db.sql("""
    SELECT name,
           composition_class AS class,
           ROUND(delta_v_km_s, 2) AS delta_v,
           ROUND(break_even_kg, 0) AS break_even_kg,
           ROUND(mission_revenue_usd, 0) AS mission_revenue,
           ROUND(mission_cost_usd, 0) AS mission_cost,
           ROUND(mission_profit_usd, 0) AS mission_profit,
           ROUND(missions_supported, 0) AS missions,
           ROUND(campaign_profit_usd, 0) AS campaign_profit
    FROM atlas
    WHERE is_viable
    ORDER BY mission_profit_usd DESC
    LIMIT 20
""")

---
## 4. Per-Metal Break-Even

How many kg of each metal to extract to cover the full mission cost.

In [ ]:
db.sql("""
    SELECT name,
           composition_class AS class,
           ROUND(delta_v_km_s, 2) AS delta_v,
           ROUND(break_even_kg, 0) AS total_be,
           ROUND(break_even_rhodium_kg, 0) AS rhodium_be,
           ROUND(break_even_gold_kg, 0) AS gold_be,
           ROUND(break_even_iridium_kg, 0) AS iridium_be,
           ROUND(break_even_platinum_kg, 0) AS platinum_be,
           ROUND(extractable_rhodium_kg, 1) AS rhodium_avail,
           ROUND(extractable_gold_kg, 1) AS gold_avail,
           ROUND(extractable_iridium_kg, 1) AS iridium_avail,
           ROUND(extractable_platinum_kg, 1) AS platinum_avail,
           is_viable
    FROM atlas
    WHERE margin_per_kg > 0
    ORDER BY margin_per_kg DESC
    LIMIT 20
""")

---
## 5. Material Inventory — M-type Targets

In [ ]:
db.sql("""
    SELECT name,
           ROUND(diameter_estimated_km, 2) AS diameter_km,
           ROUND(delta_v_km_s, 2) AS delta_v,
           ROUND(extractable_platinum_kg, 1) AS platinum_kg,
           ROUND(extractable_palladium_kg, 1) AS palladium_kg,
           ROUND(extractable_rhodium_kg, 1) AS rhodium_kg,
           ROUND(extractable_iridium_kg, 1) AS iridium_kg,
           ROUND(extractable_gold_kg, 1) AS gold_kg,
           ROUND(total_precious_value_usd, 0) AS total_value,
           is_viable,
           ROUND(missions_supported, 0) AS missions
    FROM atlas
    WHERE composition_class = 'M' AND margin_per_kg > 0
    ORDER BY total_precious_value_usd DESC
    LIMIT 20
""")

---
## 6. Break-Even by Delta-v Range

In [ ]:
db.sql("""
    SELECT
        CASE
            WHEN delta_v_km_s < 1 THEN '<1'
            WHEN delta_v_km_s < 2 THEN '1-2'
            WHEN delta_v_km_s < 3 THEN '2-3'
            WHEN delta_v_km_s < 4 THEN '3-4'
            WHEN delta_v_km_s < 5 THEN '4-5'
            ELSE '5+'
        END AS dv_range,
        COUNT(*) AS asteroids,
        ROUND(AVG(margin_per_kg), 0) AS avg_margin,
        ROUND(AVG(break_even_kg), 0) AS avg_break_even,
        COUNTIF(is_viable) AS viable,
        ROUND(SUM(CASE WHEN is_viable THEN campaign_profit_usd ELSE 0 END), 0) AS total_profit
    FROM atlas
    WHERE margin_per_kg > 0
    GROUP BY 1
    ORDER BY 1
""")

---
## 7. NEO Targets

In [ ]:
db.sql(f"""
    SELECT {STD_COLS}
    FROM atlas
    WHERE neo = 'Y' AND is_viable
    ORDER BY campaign_profit_usd DESC
    LIMIT 30
""")

---
## 8. Profitability by Composition

In [ ]:
db.sql("""
    SELECT composition_class AS class,
           COUNT(*) AS count,
           COUNTIF(margin_per_kg > 0) AS positive_margin,
           COUNTIF(is_viable) AS viable,
           ROUND(SUM(CASE WHEN is_viable THEN campaign_profit_usd ELSE 0 END), 0) AS total_campaign_profit,
           ROUND(SUM(CASE WHEN is_viable THEN missions_supported ELSE 0 END), 0) AS total_missions
    FROM atlas
    WHERE economic_score IS NOT NULL
    GROUP BY composition_class
    ORDER BY total_campaign_profit DESC
""")

---
## 9. Profitability by Orbit Class

In [ ]:
db.sql("""
    SELECT orbit_class,
           COUNT(*) AS count,
           ROUND(AVG(delta_v_km_s), 2) AS avg_delta_v,
           COUNTIF(margin_per_kg > 0) AS positive_margin,
           COUNTIF(is_viable) AS viable,
           ROUND(SUM(CASE WHEN is_viable THEN campaign_profit_usd ELSE 0 END), 0) AS total_profit
    FROM atlas
    WHERE orbit_class IS NOT NULL
    GROUP BY orbit_class
    ORDER BY viable DESC
    LIMIT 15
""")

---
## 10. Delta-v Distribution

In [ ]:
db.delta_v_histogram(bin_width=1.0)

---
## 11. Custom Query

In [ ]:
db.sql(f"""
    SELECT {STD_COLS}
    FROM atlas
    LIMIT 10
""")

---
## 12. Horizons-Enhanced NEAs

NEA targets where Horizons high-precision elements were used instead of SBDB 2-body approximations.

In [ ]:
db.sql("""
    SELECT
        orbital_precision_source AS source,
        COUNT(*) AS count,
        ROUND(AVG(delta_v_km_s), 2) AS avg_delta_v,
        ROUND(MIN(delta_v_km_s), 2) AS min_delta_v,
        COUNTIF(margin_per_kg > 0) AS positive_margin,
        COUNTIF(is_viable) AS viable
    FROM atlas
    WHERE neo = 'Y'
    GROUP BY orbital_precision_source
    ORDER BY source
""")

---
## 13. Composition Source Breakdown

How each classification layer contributes to reducing "unknown" asteroids.

In [ ]:
db.sql("""
    SELECT
        composition_source AS source,
        composition_class AS class,
        COUNT(*) AS count,
        ROUND(AVG(resource_value_usd_per_kg), 2) AS avg_value_per_kg,
        COUNTIF(is_viable) AS viable_targets
    FROM atlas
    GROUP BY composition_source, composition_class
    ORDER BY composition_source, count DESC
""")

In [ ]:
db.close()